In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
 
raw = pd.read_parquet("data/train-00000-of-00001.parquet")
 
classification = {
    "appID": "Identification",
    "name": "NonApplicable",
    "release_date": "Applicable",
    "estimated_owners": "Leakage",
    "peak_ccu": "Leakage",
    "required_age": "Applicable",
    "price": "Applicable",
    "dlc_count": "Leakage",
    "detailed_description": "Applicable (difficult)",
    "short_description": "Applicable (difficult)",
    "supported_languages": "Applicable",
    "full_audio_languages": "Applicable",
    "reviews": "Leakage",
    "header_image": "NonApplicable",
    "website": "NonApplicable",
    "support_url": "NonApplicable",
    "support_email": "NonApplicable",
    "windows": "Applicable",
    "mac": "Applicable",
    "linux": "Applicable",
    "metacritic_score": "Leakage",
    "metacritic_url": "NonApplicable",
    "user_score": "Leakage",
    "positive": "Leakage",
    "negative": "Leakage",
    "score_rank": "Leakage",
    "achievements": "NonApplicable",
    "recommendations": "Leakage",
    "notes": "Applicable (difficult)",
    "average_playtime_forever": "Leakage",
    "average_playtime_2weeks": "Leakage",
    "median_playtime_forever": "Leakage",
    "median_playtime_2weeks": "Leakage",
    "developers": "Applicable",
    "publishers": "Applicable",
    "categories": "Applicable",
    "genres": "Applicable",
    "tags": "NonApplicable / Leakage",
    "screenshots": "NonApplicable",
    "movies": "NonApplicable",
    "packages": "NonApplicable",
}
 
columns_overview = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "classification": pd.Series(classification),
})
usable_cols = columns_overview.index[
    ~columns_overview["classification"].str.contains("NonApplicable")
]

In [38]:
software_genres = {
    "Game Development", "Design & Illustration", "Animation & Modeling",
    "Utilities", "Video Production", "Software Training", "Audio Production",
    "Photo Editing", "Web Publishing", "Accounting", "Education",
}
 
def is_software(genres):
    genres = set(genres) if genres is not None else set()
    return len(genres) == 0 or genres <= software_genres
 
df = raw[usable_cols].copy()
df = df[~df["genres"].apply(is_software)].copy()
 
df["language_count"] = df["supported_languages"].str.len().fillna(0).astype(int)
df["audio_count"] = df["full_audio_languages"].str.len().fillna(0).astype(int)
df["self_published"] = df.apply(
    lambda r: len(set(r["developers"]) & set(r["publishers"])) > 0, axis=1
)
 
df["log_positive"] = np.log1p(df["positive"])
target = "log_positive"


In [40]:
df_tmp, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_tmp, test_size=0.25, random_state=42)
len(df_train), len(df_val), len(df_test)

(68712, 22905, 22905)

In [52]:
def val_score(features):
    m = LinearRegression().fit(df_train[features], df_train[target])
    p = m.predict(df_val[features])
    return r2_score(df_val[target], p), mean_absolute_error(df_val[target], p)

baseline = DummyRegressor(strategy="mean")
baseline.fit(df_train[["price"]], df_train[target])
pred = baseline.predict(df_val[["price"]])
baseline_mae = mean_absolute_error(df_val[target], pred)
baseline_r2 = r2_score(df_val[target], pred)
 
start_features = ["price", "language_count", "self_published", "linux", "mac", "audio_count", "dev_game_count"]
start_r2, start_mae = val_score(start_features)
 
print("\nTask 1: validation results (log scale)")
print(pd.DataFrame({
    "MAE": [baseline_mae, start_mae],
    "R²": [baseline_r2, start_r2],
}, index=["Baseline (mean)", "Start model"]).round(3).to_string())


Task 1: validation results (log scale)
                   MAE     R²
Baseline (mean)  1.948 -0.000
Start model      1.886  0.052


En hel del av features från Milestone 2 samt några till som testas.
Endast mer populära genres och categories

In [53]:
def as_set(x):
    return set(x) if isinstance(x, (list, tuple, np.ndarray)) else set()
 
dev_count = df_train["developers"].explode().value_counts()
 
min_games = 0.02 * len(df_train)
top_genres = df_train["genres"].explode().value_counts()
top_genres = top_genres[top_genres >= min_games].index
top_cats = df_train["categories"].explode().value_counts()
top_cats = top_cats[top_cats >= min_games].index
 
for d in (df_train, df_val, df_test):
    d["log_price"] = np.log1p(d["price"])
    d["is_free"] = (d["price"] == 0).astype(int)
    d["log_languages"] = np.log1p(d["language_count"])
    d["log_audio"] = np.log1p(d["audio_count"])
    d["dev_game_count"] = d["developers"].apply(
        lambda devs: max((dev_count.get(x, 0) for x in as_set(devs)), default=0))
    d["log_dev_games"] = np.log1p(d["dev_game_count"])
    d["mature"] = (d["required_age"] > 0).astype(int)
    for c in ["windows", "mac", "linux", "self_published"]:
        d[c] = d[c].astype(int)
    for g in top_genres:
        d["genre_" + g] = d["genres"].apply(lambda x: int(g in as_set(x)))
    for c in top_cats:
        d["cat_" + c] = d["categories"].apply(lambda x: int(c in as_set(x)))
 
genre_cols = ["genre_" + g for g in top_genres]
cat_cols = ["cat_" + c for c in top_cats]
single = ["price", "log_price", "is_free", "language_count", "log_languages",
          "audio_count", "log_audio", "dev_game_count", "log_dev_games",
          "self_published", "windows", "mac", "linux", "mature"]
candidates = single + genre_cols + cat_cols

In [55]:
corr = df_train[candidates].corrwith(df_train[target])

print("Correlation with", target)
print(corr.round(2).sort_values(key=abs, ascending=False).to_string())

Correlation with log_positive
cat_Steam Trading Cards             0.43
cat_Steam Cloud                     0.26
cat_Steam Achievements              0.23
cat_Remote Play on TV               0.19
cat_Full controller support         0.18
self_published                     -0.17
log_languages                       0.17
mature                              0.17
cat_Steam Workshop                  0.16
mac                                 0.15
genre_Casual                       -0.15
cat_Multi-player                    0.14
cat_Co-op                           0.13
log_price                           0.12
genre_Free To Play                 -0.11
cat_Playable without Timed Input   -0.11
cat_Online Co-op                    0.11
cat_Custom Volume Controls         -0.10
cat_Family Sharing                  0.10
log_audio                           0.10
is_free                            -0.10
cat_Includes level editor           0.10
cat_Steam Leaderboards              0.09
cat_Partial Controller Supp

En hel del olika features som ska testas. Därför en loop som ser om en feature har en rikgtig påverkan på R² validation.

In [ ]:
pairs = {"price": "log_price", "language_count": "log_languages",
         "audio_count": "log_audio", "dev_game_count": "log_dev_games"}
groups = [[(raw_f, [raw_f]), (log_f, [log_f])] for raw_f, log_f in pairs.items()]
groups += [[(f, [f])] for f in ["is_free", "self_published", "windows", "mac", "linux", "mature"]]
groups += [[("genres", genre_cols)], [("categories", cat_cols)]]
 
def strength(group):
    cols = [c for _, cs in group for c in cs]
    return np.nan_to_num(df_train[cols].corrwith(df_train[target]).abs().max())
 
groups.sort(key=strength, reverse=True)
 
features, steps = [], []
best_r2 = baseline_r2
for group in groups:
    results = [(name, cols) + val_score(features + cols) for name, cols in group]
    best = max(results, key=lambda r: r[2])
    keep = best[2] - best_r2 >= 0.003
    for name, cols, r2, mae in results:
        steps.append({"added": name, "MAE val": round(mae, 3), "R² val": round(r2, 3),
                      "kept": "yes" if keep and name == best[0] else "no"})
    if keep:
        features += best[1]
        best_r2 = best[2]
 
print("One feature at a time (validation, log scale)")
print(pd.DataFrame(steps).to_string(index=False))
 
if not features:                           # nothing beat the baseline: keep the start model
    print("No candidate improved the result; the final model uses the start features.")
    features = start_features


Task 3: one feature at a time (validation, log scale)
         added  MAE val  R² val kept
    categories    1.618   0.279  yes
        mature    1.606   0.294  yes
self_published    1.591   0.303  yes
language_count    1.589   0.304   no
 log_languages    1.587   0.307  yes
           mac    1.582   0.312  yes
        genres    1.559   0.327  yes
         price    1.558   0.328   no
     log_price    1.559   0.327   no
       is_free    1.559   0.328   no
   audio_count    1.555   0.330   no
     log_audio    1.559   0.327   no
         linux    1.557   0.328   no
dev_game_count    1.559   0.327   no
 log_dev_games    1.549   0.329   no
       windows    1.559   0.327   no


In [58]:
final = LinearRegression().fit(df_train[features], df_train[target])
final_r2, final_mae = val_score(features)
 
print("Final model")
print("Validation MAE:", round(final_mae, 3), " R²:", round(final_r2, 3),
      " (start model:", round(start_mae, 3), "/", round(start_r2, 3), ")")
print("intercept:", round(final.intercept_, 2))
weights = pd.Series(final.coef_, index=features).sort_values(key=abs, ascending=False)
summary = pd.DataFrame({
    "weight": weights.round(3),
    "effect on reviews": ((np.exp(weights) - 1) * 100).round(1).astype(str) + " %",
})
print(summary.to_string())

Final model
Validation MAE: 1.559  R²: 0.327  (start model: 1.886 / 0.052 )
intercept: 1.78
                                  weight effect on reviews
cat_Steam Trading Cards            2.350           948.7 %
mature                             2.196           799.2 %
cat_Steam Workshop                 1.210           235.4 %
cat_PvP                           -1.154           -68.5 %
cat_Remote Play on TV              0.973           164.5 %
cat_Online PvP                     0.971           164.0 %
cat_Multi-player                   0.881           141.4 %
cat_Tracked Controller Support     0.770           115.9 %
cat_Custom Volume Controls        -0.754           -52.9 %
cat_Shared/Split Screen           -0.716           -51.1 %
cat_Shared/Split Screen PvP        0.689            99.1 %
cat_Playable without Timed Input  -0.685           -49.6 %
self_published                    -0.545           -42.0 %
genre_Free To Play                -0.544           -42.0 %
cat_Steam Cloud        

In [61]:
pred_val = final.predict(df_val[features])
check = df_val[["positive", "price", "genres", "developers"]].copy()
check.insert(0, "name", raw.loc[df_val.index, "name"])
check["predicted"] = np.expm1(pred_val).round(0)
check["residual"] = (df_val[target] - pred_val).round(2)
check = check.sort_values("residual")
 
print("Model guessed far too LOW (more reviews than predicted):")
print(check.tail(10).to_string())
print("\nModel guessed far too HIGH (fewer reviews than predicted):")
print(check.head(10).to_string())

Model guessed far too LOW (more reviews than predicted):
                               name  positive  price                                                        genres              developers  predicted  residual
14405   Spooky's Jump Scare Mansion     13926   0.00                      [Action, Adventure, Free To Play, Indie]           [Lag Studios]        2.0      8.55
119656                      Valheim    471451   9.99                 [Action, Adventure, Indie, RPG, Early Access]          [Iron Gate AB]       74.0      8.74
109015                Marvel Rivals    228817   0.00                                        [Action, Free To Play]         [NetEase Games]       22.0      9.21
87783                   The Sims™ 4    149064   0.00                 [Adventure, Casual, Simulation, Free To Play]                 [Maxis]       14.0      9.24
100399                    ULTRAKILL    165867  17.49                                 [Action, Indie, Early Access]  [Arsi "Hakita" Patala]     

In [48]:
pred_test = final.predict(df_test[features])
print("R² test (log scale):", round(r2_score(df_test[target], pred_test), 3))
print("MAE test (log scale):", round(mean_absolute_error(df_test[target], pred_test), 3))
 
back = np.expm1(pred_test)
print("Median abs. error (reviews):", round(np.median(np.abs(df_test["positive"] - back)), 0))

R² test (log scale): 0.328
MAE test (log scale): 1.542
Median abs. error (reviews): 8.0
